# 水文相似性指标体系的构建研究

## 预处理

In [1]:
from pickle import load

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler

from watershed import *
from similarity import *
from model import *

In [2]:
# 数据归一化处理

all_features_S = []
all_features_H = []
for i in range(1, 26):
    if i != 18:
        with open("./cache/%d_r=90" % i, "rb") as f:
            example = load(f)
            all_features_S.append(example.feature_S().reshape(-1))
            all_features_H.append(example.feature_H().reshape(-1))
all_features_S = np.array(all_features_S)
all_features_H = np.array(all_features_H)

# 0-1归一化
minmax_S = MinMaxScaler().fit(all_features_S)
minmax_H = MinMaxScaler().fit(all_features_H)
# 正态归一化
norm_S = StandardScaler().fit(all_features_S)
norm_H = StandardScaler().fit(all_features_H)

训练数据：1，2，3，4，5，10，11，12，13，19，20，21

测试数据：6，7，8，9，14，15，16，17，22，23，24，25

In [ ]:
train_set = []
valid_set = []

for id in range(1, 6):
    with open("./cache/%d_r=90" % id, "rb") as f:
        train_set.append(load(f))

for id in range(10, 14):
    with open("./cache/%d_r=90" % id, "rb") as f:
        train_set.append(load(f))

for id in range(19, 22):
    with open("./cache/%d_r=90" % id, "rb") as f:
        train_set.append(load(f))

for id in range(6, 10):
    with open("./cache/%d_r=90" % id, "rb") as f:
        valid_set.append(load(f))
for id in range(14, 18):
    with open("./cache/%d_r=90" % id, "rb") as f:
        valid_set.append(load(f))
for id in range(22, 26):
    with open("./cache/%d_r=90" % id, "rb") as f:
        valid_set.append(load(f))

size_train = len(train_set)
size_valid = len(valid_set)
print("train_set size: %d; valid_set size: %d" % (size_train, size_valid))

In [4]:
S_name = ["height", "density", "slope", "TWI", "clay", "sand", "silt", "NDVI trend", "NDVI s1", "NDVI s2", "NDVI s3", "NDVI s4", "cropland", "urban"]
H_name = ["runoff", "baseflow", "recession_early", "recession_custom", "recession_late"]

## 主成分分析

In [5]:
model = SimilarityModel(quotient, quotient, minmax_S, minmax_H)

In [ ]:
sns.set_style("darkgrid")
sns.set_palette("colorblind")

for k in range(5):
    fig, ax = plt.subplots(figsize=(8, 6), dpi=200)
    y = np.zeros(shape=(14, 120))
    alphas = np.arange(0.001, 0.121, 0.001)
    for i, alpha in enumerate(alphas):
        model.fit(train_set, train_set, alpha, "Lasso")
        y[:, i] = model.core.coef_[k]

    for j in range(14):
        if np.mean(y[j]) > 1e-2:
            sns.lineplot(x=alphas, y=y[j], label=S_name[j])

    ax.set_title("Lasso Principal Component Analysis\n of [%s]" % H_name[k])
    ax.set_xlabel("alpha", fontsize=12)
    ax.set_ylabel("coefficient", fontsize=12)
    plt.legend(loc='upper right')
    plt.show()

In [ ]:
sns.set_style("darkgrid")
sns.set_palette("colorblind")

for k in range(5):
    fig, ax = plt.subplots(figsize=(8, 6), dpi=200)
    y = np.zeros(shape=(14, 100))
    alphas = np.arange(0.21, 1.21, 0.01)
    for i, alpha in enumerate(alphas):
        model.fit(train_set, train_set, alpha, "Ridge")
        y[:, i] = model.core.coef_[k]

    for j in range(14):
        if np.mean(y[j]) > 1e-2:
            sns.lineplot(x=alphas, y=y[j], label=S_name[j])

    ax.set_title("Ridge Principal Component Analysis\n of [%s]" % H_name[k])
    ax.set_xlabel("alpha", fontsize=12)
    ax.set_ylabel("coefficient", fontsize=12)
    plt.legend(loc='upper right')
    plt.show()


## 模型评价

In [8]:
def score_model(model:SimilarityModel, valid_set_1:List[Watershed], valid_set_2:List[Watershed], plot=False):

    Y_pred, Y_true = model.predict(valid_set_1, valid_set_2)

    Y_true = Y_true.reshape(len(valid_set_1), len(valid_set_2), 5)
    Y_pred = Y_pred.reshape(len(valid_set_1), len(valid_set_2), 5)
    grade_true = grade(Y_true)
    grade_pred = grade(Y_pred)
    grade_mean_true = grade(np.mean(Y_true, axis=2))
    grade_mean_pred = grade(np.mean(Y_pred, axis=2))

    loss = np.mean(np.abs(Y_true - Y_pred))
    accuracy_Top1 = np.sum(grade_true == grade_pred) / (len(valid_set_1) * len(valid_set_2) * 5)
    accuracy_Mean = np.sum(grade_mean_true == grade_mean_pred) / (len(valid_set_1) * len(valid_set_2))

    print("Loss    : %.6f" %  loss)
    print("Top1 acc: %.2f%%" % (accuracy_Top1 * 100))
    print("Mean acc: %.2f%%" % (accuracy_Mean * 100))

    if plot:
        fig, ax = plt.subplots(3, 2, figsize=(18, 18), dpi=400)

        x_labels = [e.id for e in valid_set_1]
        y_labels = [e.id for e in valid_set_2]

        sns.heatmap(np.mean(Y_true, axis=2), ax=ax[0, 0], cmap=sns.cm.rocket_r, xticklabels=x_labels, yticklabels=y_labels)
        ax[0, 0].set_title("Similarity (True)")
        sns.heatmap(np.mean(Y_pred, axis=2), ax=ax[1, 0], cmap=sns.cm.rocket_r, xticklabels=x_labels, yticklabels=y_labels)
        ax[1, 0].set_title("Similarity (Pred)")
        sns.heatmap(np.abs(np.mean(Y_true - Y_pred, axis=2)), ax=ax[2, 0], cmap=sns.cm.rocket_r, xticklabels=x_labels, yticklabels=y_labels)
        ax[2, 0].set_title("Similarity Delta")

        sns.heatmap(grade_mean_true, ax=ax[0, 1], cmap=sns.cm.rocket_r, xticklabels=x_labels, yticklabels=y_labels)
        ax[0, 1].set_title("Grade (True)")
        sns.heatmap(grade_mean_pred, ax=ax[1, 1], cmap=sns.cm.rocket_r, xticklabels=x_labels, yticklabels=y_labels)
        ax[1, 1].set_title("Grade (Pred)")
        sns.heatmap(grade_mean_true == grade_mean_pred, ax=ax[2, 1], cmap=sns.cm.rocket, xticklabels=x_labels, yticklabels=y_labels)
        ax[2, 1].set_title("Wrong Results")

        plt.show()

- quotient

In [ ]:
model = SimilarityModel(quotient, quotient, minmax_S, minmax_H)

model.fit(train_set, train_set, 0.1, "Ridge")
score_model(model, valid_set, valid_set, True)

- delta

In [ ]:
model = SimilarityModel(delta, quotient, minmax_S, minmax_H)

model.fit(train_set, train_set, 0.1, "Ridge")
score_model(model, valid_set, valid_set, True)

- exponent

In [ ]:
model = SimilarityModel(exponent, quotient, minmax_S, minmax_H)


model.fit(train_set, train_set, 0.1, "Ridge")
score_model(model, valid_set, valid_set, True)

- quadratic

In [ ]:
model = SimilarityModel(quadratic, quotient, minmax_S, minmax_H)

model.fit(train_set, train_set, 0.1, "Ridge")
score_model(model, valid_set, valid_set, True)

## 显著性分析

In [ ]:
model = SimilarityModel(delta, quotient, minmax_S, minmax_H)

model.fit(train_set, train_set, 0.1, "Ridge")
score_model(model, valid_set, valid_set)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6), dpi=400)

sns.heatmap(np.abs(model.core.coef_), annot=True, fmt=".2f", xticklabels=S_name, yticklabels=H_name, cmap=sns.cm.rocket_r)
ax.set_title("Significance Analysis")

plt.show()